# FTIS Exploratory Data Analysis

This notebook explores engineered turbulence features, FTI distribution, altitude effects, wind correlations, and quick feature importance diagnostics.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ftis.config import LABEL_ORDER, NUMERIC_MODEL_FEATURES, TARGET_COLUMN
from ftis.features import engineer_turbulence_features

FEATURES_PATH = ROOT / 'data' / 'processed' / 'features.csv'
TRAINING_PATH = ROOT / 'data' / 'processed' / 'training_data.csv'

if FEATURES_PATH.exists():
    df = pd.read_csv(FEATURES_PATH)
else:
    df = engineer_turbulence_features(pd.read_csv(TRAINING_PATH))

df.head()

## Turbulence Distribution

In [ ]:
label_counts = df[TARGET_COLUMN].value_counts().reindex(LABEL_ORDER, fill_value=0)
fig = px.bar(
    label_counts.reset_index(),
    x=TARGET_COLUMN,
    y='count',
    color=TARGET_COLUMN,
    title='Turbulence Label Distribution',
    color_discrete_map={'Low': '#22c55e', 'Moderate': '#facc15', 'High': '#ef4444'},
)
fig.show()

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df['FTI'], bins=30, color='#38bdf8', edgecolor='white')
plt.title('Flight Turbulence Index Distribution')
plt.xlabel('FTI')
plt.ylabel('Count')
plt.grid(alpha=0.25)
plt.show()

## Altitude and Wind Relationships

In [ ]:
fig = px.scatter(
    df,
    x='altitude',
    y='FTI',
    color=TARGET_COLUMN,
    hover_data=['latitude', 'longitude', 'windspeed', 'pressure', 'temperature'],
    title='Altitude vs Flight Turbulence Index',
    color_discrete_map={'Low': '#22c55e', 'Moderate': '#facc15', 'High': '#ef4444'},
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='windspeed',
    y='FTI',
    color='altitude_band',
    trendline='ols',
    title='Windspeed Correlation with FTI',
)
fig.show()

## Feature Correlation Heatmap

In [ ]:
heatmap_features = [
    'altitude', 'windspeed', 'pressure', 'temperature', 'wind_shear',
    'temperature_gradient', 'pressure_variation',
    'atmospheric_instability_score', 'turbulence_intensity_score', 'FTI'
]
corr = df[heatmap_features].corr(numeric_only=True)
fig = px.imshow(
    corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    title='FTIS Feature Correlation Heatmap',
)
fig.show()

## Quick Feature Importance Diagnostic

In [ ]:
model_df = df.dropna(subset=[*NUMERIC_MODEL_FEATURES, TARGET_COLUMN]).copy()
X = model_df[NUMERIC_MODEL_FEATURES]
y = model_df[TARGET_COLUMN].astype(str).str.title().map({label: idx for idx, label in enumerate(LABEL_ORDER)})

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced_subsample')
rf.fit(X, y)
importance = pd.DataFrame({'feature': NUMERIC_MODEL_FEATURES, 'importance': rf.feature_importances_})
importance = importance.sort_values('importance', ascending=False)

fig = px.bar(
    importance,
    x='importance',
    y='feature',
    orientation='h',
    title='Random Forest Feature Importance',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()